### This notebook compute "P13. Water yield: Water production per COMID and per square kilometer" indicator for the 27 basins of IKI Project

**Created:** 12/17/2025 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 12/22/2025 by Sophia Bakar

**Status:** Complete for baseline scenario and first future scenario

**QA Status:** reviewed by  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Peligro
 
**Objective:**   

**Compatibility:** 

**Packages:** numpy, pandas, geopandas, sqlite3, os, re  

**Further documentation:**  
 
**Inputs:** area of each COMID (source: subbasins shapfile); annual average of water production for each COMID (source: waterALLOC output database)

**Outputs:** 
 
**Assumptions:** We do not normalize the results in this script because we normalize them as the first step of the impact chains calculation.

**N/A Handling:** No missing data.  

**Future work:** 
 
**Notes:** For this indicator, we get the annual average of water production for each COMID from the waterALLOC output database. Then we divide that by the area in square kilometers for each COMID. When we do the normalization step, we need to normalize by the min and max across all the COMIDs. In this script, we include code to check the min and max of our results and the min and max that is listed in the indicators database to ensure they align.  

General methodology:  
1. Pull the area (km2) for each COMID.
2. For each scenario, extract the annual average water production for each COMID.
3. Divide the annual average water production by COMID area.

In [1]:
import numpy as np
import pandas as pd
import sqlite3
import geopandas as gpd
import os
import re
import matplotlib.pyplot as plt
import itertools

In [2]:
user = 'sbakar'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
# wateralloc_db = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"
wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

In [3]:
# Set up indicator ID
IndID = 113  # Indicator ID (Exposure = 2 + 0X, Peligro = 1 +0x, etc.)

# Connect to Indicators DB and get available scenarios
conn = sqlite3.connect(db_path)

wa_scenarios_df = pd.read_sql_query(
    """
    SELECT WaScnID, WaScnName
    FROM WaScenarios
    ORDER BY WaScnID
    """,
    conn
)

conn.close()

# For now: only baseline and first future
scenario_ids = wa_scenarios_df.loc[
    wa_scenarios_df['WaScnID'].isin([1, 3]), 'WaScnID'
].tolist()

# for all scenarios:
#scenario_ids = wa_scenarios_df['WaScnID'].tolist()

In [4]:
wa_scenarios_df

,WaScnID,WaScnName
0,1,CC_CMIP6_85_2050
1,3,Linea_Base_2020
2,4,Linea_Base_2020_Embalses


In [5]:
#subbasins_shapefile = f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/GIS_WaterALLOC_General/Peru_AHD_with_districts.shp'
subbasins_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
subbasins_gdf = gpd.read_file(subbasins_shapefile).set_index('COMID').to_crs('WGS84')

In [6]:
# Loop through all scenarios and fetch OfertaTot data
oferta_all = []

conn_wa = sqlite3.connect(wateralloc_db)

for scn_id in scenario_ids:
    print(f"\nProcessing scenario: (WaScnID={scn_id})")

    query_oferta = """
    SELECT
        COMID,
        AVG(OfertaTot_anual) AS OfertaTot_mean,
        COUNT(Año) AS n_years
    FROM (
        SELECT
            a.comid AS COMID,
            a.Año,
            SUM(a.Afluencia) AS OfertaTot_anual
        FROM [WAMSS_Oferta anual por tipo por COMID] AS a
        JOIN WAMMS_RunsInfo AS b
            ON a.RunID = b.RunID
        JOIN Scenarios AS c
            ON c.WaScnID = b.WaScnID
        WHERE c.WaScnID = ?
          AND a.TipoAfluencia <> 'Recarga'
        GROUP BY a.comid, a.Año
    )
    GROUP BY COMID
    """

    oferta_df = pd.read_sql_query(
        query_oferta,
        conn_wa,
        params=(scn_id,)
    )

    # Add dynamic scenario ID
    oferta_df['WaScnID'] = scn_id

    # Merge basin area
    area_df = (
        subbasins_gdf[['AREASQKM']]
        .reset_index()
        .rename(columns={'AREASQKM': 'Area_km2'})
    )

    oferta_df = oferta_df.merge(area_df, on='COMID', how='left')

    # Normalize by area
    oferta_df['OfertaTot_mean_norm'] = (
        oferta_df['OfertaTot_mean'] / oferta_df['Area_km2']
    )

    oferta_all.append(oferta_df)

conn_wa.close()

oferta_all_df = pd.concat(oferta_all, ignore_index=True)



Processing scenario: (WaScnID=1)

Processing scenario: (WaScnID=3)


In [7]:
# Create full COMID x scenario combinations (Cartesian product)
all_comids = subbasins_gdf.reset_index()[['COMID']]
all_scenarios = pd.DataFrame({'WaScnID': scenario_ids})

all_comid_scenarios = pd.DataFrame(
    list(itertools.product(all_comids['COMID'], all_scenarios['WaScnID'])),
    columns=['COMID', 'WaScnID']
)

# Merge Oferta data in
oferta_gdf = all_comid_scenarios.merge(
    oferta_all_df,
    on=['COMID', 'WaScnID'],
    how='left'
)

# Merge geometry and area
subbasins_reset = subbasins_gdf.reset_index()[['COMID', 'geometry', 'AREASQKM']]
oferta_gdf = oferta_gdf.merge(
    subbasins_reset,
    on='COMID',
    how='left'
).rename(columns={'AREASQKM': 'Area_km2'})

# Recalculate normalized value if missing
if 'OfertaTot_mean_norm' not in oferta_gdf.columns:
    oferta_gdf['OfertaTot_mean_norm'] = oferta_gdf['OfertaTot_mean'] / oferta_gdf['Area_km2']

# Quick check
print(f"Total rows (COMID x scenario): {len(oferta_gdf)}")
print(f"Rows with data: {oferta_gdf['OfertaTot_mean_norm'].notna().sum()}")
print(f"Rows without data: {oferta_gdf['OfertaTot_mean_norm'].isna().sum()}")

Total rows (COMID x scenario): 7310
Rows with data: 1076
Rows without data: 6234


In [8]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
delete_query = """
DELETE FROM IndValues_WaALLOC
WHERE IndID = ?;
"""
cursor.execute(delete_query, (IndID,))
conn.commit()
print(f"Deleted existing rows for IndID = {IndID}")
conn.close()

Deleted existing rows for IndID = 113


In [9]:
# Connect to SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

rows_to_insert = []

for _, row in oferta_gdf.iterrows():
    rows_to_insert.append((
        row['WaScnID'],
        IndID,
        row['COMID'],
        row['OfertaTot_mean_norm']  # NaN -> NULL in SQLite
    ))

# Insert data into IndValues_WaALLOC
insert_query = """
INSERT OR REPLACE INTO IndValues_WaALLOC (WaScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""

In [10]:
# Check that min and max values match the expected range based on the Indicators Table

indicator_limits = pd.read_sql_query(
    """
    SELECT IndID, Min, Max
    FROM Indicators
    WHERE IndID = ?
    """,
    conn,
    params=(IndID,)
)

if indicator_limits.empty:
    raise ValueError(f"No entry found in Indicators table for IndID = {IndID}")

ind_min = indicator_limits.loc[0, 'Min']
ind_max = indicator_limits.loc[0, 'Max']

print(f"\nIndicator {IndID} limits from Indicators table -> Min: {ind_min}, Max: {ind_max}")

# Compute value stats by scenario
value_stats = oferta_all_df.groupby('WaScnID')['OfertaTot_mean_norm'].agg(['min', 'max', 'count']).reset_index()
print("\n=== Values to be inserted (by scenario) ===")
print(value_stats)

# Check for duplicates in the rows to insert
df_check = pd.DataFrame(rows_to_insert, columns=['WaScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['WaScnID', 'IndID', 'COMID'])
print("\nDuplicates in rows_to_insert:")
print(df_check[duplicates])



Indicator 113 limits from Indicators table -> Min: 172.74141234862645, Max: 1486.4266474910974

=== Values to be inserted (by scenario) ===
   WaScnID        min          max  count
0        1  36.179909  5584.168618    538
1        3  42.161279  5635.481295    538

Duplicates in rows_to_insert:
Empty DataFrame
Columns: [WaScnID, IndID, COMID, Value]
Index: []


In [13]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()